# Ce notebook servira à explorer le jeu de données brute
## Les données brutes se situeront dans ...\racine du repo\data\raw

In [ ]:
import sys

# add root path, to import all modules
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    
from src.accidents.data import fetch_collisions, load_collisions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

## Telecharge le CSV si pas deja present dans data/raw/

In [ ]:
# force=True pour forcer le re-telechargement
path = fetch_collisions(raw_dir='data/raw', force=False)

In [ ]:
df = load_collisions(path)
print(f'Shape : {df.shape}')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info(show_counts=True)

#### On voit bien quelques petites limitations, beaucoup de NaN, aussi le jeu de donnée s'arrête en 2021 (il est mentioné dans la description sur la page de la ville que si l'on veut les données plus récente il faut les chercher ici https://www.donneesquebec.ca/recherche/fr/dataset/rapports-d-accident mais ça s'arrête aussi en 2022)

In [ ]:
missing = (
    pd.DataFrame({
        'n_manquant': df.isnull().sum(),
        'pct_manquant': (df.isnull().mean() * 100).round(2)
    })
    .sort_values('pct_manquant', ascending=False)
)

print(f'{(missing["n_manquant"] > 0).sum()} colonnes avec des valeurs manquantes sur {df.shape[1]}')
missing[missing['n_manquant'] > 0]

In [ ]:
cols_with_missing = missing[missing['n_manquant'] > 0]

fig, ax = plt.subplots(figsize=(10, max(3, len(cols_with_missing) * 0.4)))
ax.barh(cols_with_missing.index, cols_with_missing['pct_manquant'], color='steelblue')
ax.axvline(60, color='red', linestyle='--', linewidth=1, label='seuil 60%') # j'ai mis un seuil pour l'instant mais j'ai pas encore établi de stratégie pour le nettoyage, reste à changer
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Valeurs manquantes par colonne (dataset brut)')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    'dtype':    df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'pct_nan':  (df.isnull().mean() * 100).round(2),
    'exemples': [str(df[c].dropna().unique()[:4].tolist()) for c in df.columns]
})
summary

## Vu qu'on compte étudier la gravité des accidents analysons un peu sa colonne

In [ ]:
COL_GRAVITE = 'GRAVITE'

counts = df[COL_GRAVITE].value_counts(dropna=False).sort_index()
pct    = (counts / len(df) * 100).round(2)

pd.DataFrame({'n': counts, '%': pct})

In [ ]:
GRAVITE_LABELS = { # pour rendre les graphiques lisibles, sinon texte bcp trop long
    'Mortel':                        'Mortel',
    'Blessures graves':              'Bl. graves',
    'Blessures légères':             'Bl. legeres',
    'Dommages matériels seulement':  'DMS',
    'Dommages matériels inférieurs au seuil de rapportage':  'DMISR',
}

labels_courts = counts.index.map(lambda x: GRAVITE_LABELS.get(x, x))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribution de la gravite (valeurs brutes)', fontsize=12)

colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

# Barplot
axes[0].bar(labels_courts, counts.values, color=colors[:len(counts)])
axes[0].set_title('Effectifs')
axes[0].set_xlabel('Gravite')
axes[0].set_ylabel('Nombre de collisions')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, (v, p) in enumerate(zip(counts.values, pct.values)):
    axes[0].text(i, v * 1.01, f'{p:.1f}%', ha='center', fontsize=10, fontweight='bold')

# pie chart
axes[1].pie(
    counts.values,
    labels=labels_courts,
    autopct='%1.1f%%',
    colors=colors[:len(counts)],
    startangle=90,
    pctdistance=0.75,
)
axes[1].set_title('Proportions')

plt.tight_layout()
plt.show()

In [ ]:
COL_DATE  = 'DT_ACCDN'
COL_HEURE = 'HEURE_ACCDN'

print('Exemples de valeurs brutes :')
print('Date  :', df[COL_DATE].dropna().unique()[:6])
print('Heure :', df[COL_HEURE].dropna().unique()[:6])

In [ ]:
annees = df[COL_DATE].astype(str).str[:4]
annees_counts = annees.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(annees_counts.index, annees_counts.values, color='steelblue')
ax.set_title('Nombre de collisions par annee (brut)')
ax.set_xlabel('Annee')
ax.set_ylabel('Nombre de collisions')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Variables categorielles contextuelles a explorer, on les mets simplement manuellement ici
CAT_COLS = [
    'JR_SEMN_ACCDN',      # jour de la semaine
    'CD_GENRE_ACCDN',     # genre de collision
    'CD_ETAT_SURFC',      # etat de la surface
    'CD_ECLRM',           # eclairage
    'CD_ENVRN_ACCDN',     # environnement (urbain/rural)
    'CD_CATEG_ROUTE',     # categorie de route
    'CD_ASPCT_ROUTE',     # aspect de la route
    'CD_LOCLN_ACCDN',     # localisation longitudinale
    'CD_CONFG_ROUTE',     # configuration de la route
    'CD_COND_METEO',      # conditions meteorologiques
    'TP_REPRR_ACCDN',     # type de repere
    'GRAVITE',            # variable cible
    'REG_ADM',            # region administrative
]

In [ ]:
for col in CAT_COLS:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False).to_string())

In [ ]:
# quelques statistiques vite fait pour nos variables numériques
num_df = df.select_dtypes(include=[np.number])
print(f'{num_df.shape[1]} variables numeriques :')
num_df.describe().round(2)

In [ ]:
COL_LAT = 'LOC_LAT'
COL_LON = 'LOC_LONG'

if COL_LAT in df.columns and COL_LON in df.columns:
    lat = pd.to_numeric(df[COL_LAT], errors='coerce')
    lon = pd.to_numeric(df[COL_LON], errors='coerce')

    print(f'Latitude  — min: {lat.min():.4f}  max: {lat.max():.4f}  NaN: {lat.isna().sum():,}')
    print(f'Longitude — min: {lon.min():.4f}  max: {lon.max():.4f}  NaN: {lon.isna().sum():,}')

    valides = lat.notna() & lon.notna()
    print(f'\nPoints avec coordonnees valides : {valides.sum():,} / {len(df):,} ({valides.mean()*100:.1f}%)')
else:
    print(f'Colonnes {COL_LAT} ou {COL_LON} introuvables — verifier les noms.')

In [ ]:
sample = df[valides].sample(min(20_000, valides.sum()), random_state=42)
lat_s = pd.to_numeric(sample[COL_LAT], errors='coerce')
lon_s = pd.to_numeric(sample[COL_LON], errors='coerce')

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(lon_s, lat_s, alpha=0.05, s=1, color='crimson')
ax.set_title(f'Distribution spatiale brute (echantillon {len(sample):,} points)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

# Conclusion
Maintenant qu'on a fait une petite exploration du jeu de données brutes, on a une meilleure idée du nettoyage que nous devons faire Pour faire bref, il faut traiter les nombreux NaN, encoder les variables catégorielles, standardiser. Le nettoyage se fera dans son propre notebook et on refera une exploration pour comparer les données avant/après nettoyage

---

# Post-Nettoyage
Maintenant le nettoyage de base fait (et expliqué) dans `data_cleaning.ipynb`, il est possible de comparer les données de base et celles nettoyé dans quelques différentes facettes.

In [ ]:
df_clean = load_collisions(Path('../data/clean/collisions_clean.csv'))
print(f'Données brutes   : {df.shape}')
print(f'Données nettoyées: {df_clean.shape}')
print(f'Lignes supprimées: {df.shape[0] - df_clean.shape[0]:,}')
print(f'Colonnes (avant) : {df.shape[1]}')
print(f'Colonnes (après) : {df_clean.shape[1]}')


## Comparaison avant/après nettoyage : Valeurs manquantes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Avant nettoyage
missing_before = df.isnull().sum()
missing_before_pct = (df.isnull().mean() * 100).round(2)
cols_with_missing_before = missing_before[missing_before > 0].sort_values(ascending=False)

axes[0].barh(range(len(cols_with_missing_before)), missing_before_pct.loc[cols_with_missing_before.index], color='coral')
axes[0].set_yticks(range(len(cols_with_missing_before)))
axes[0].set_yticklabels(cols_with_missing_before.index, fontsize=9)
axes[0].set_xlabel('% de valeurs manquantes')
axes[0].set_title('AVANT nettoyage - Valeurs manquantes par colonne')
axes[0].invert_yaxis()
axes[0].axvline(60, color='red', linestyle='--', linewidth=1, alpha=0.5, label='seuil 60%')

# Après nettoyage
missing_after = df_clean.isnull().sum()
missing_after_pct = (df_clean.isnull().mean() * 100).round(2)
cols_with_missing_after = missing_after[missing_after > 0].sort_values(ascending=False)

if len(cols_with_missing_after) > 0:
    axes[1].barh(range(len(cols_with_missing_after)), missing_after_pct.loc[cols_with_missing_after.index], color='lightgreen')
    axes[1].set_yticks(range(len(cols_with_missing_after)))
    axes[1].set_yticklabels(cols_with_missing_after.index, fontsize=9)
else:
    axes[1].text(0.5, 0.5, '✓ Aucune valeur manquante !', 
                ha='center', va='center', fontsize=14, fontweight='bold', color='green')
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)

axes[1].set_xlabel('% de valeurs manquantes')
axes[1].set_title('APRÈS nettoyage - Valeurs manquantes par colonne')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Colonnes avec NaN avant : {(df.isnull().sum() > 0).sum()}')
print(f'Colonnes avec NaN après : {(df_clean.isnull().sum() > 0).sum()}')


## Comparaison : Types de données et transformations

In [ ]:
# Comparer les types de données
print("=" * 70)
print("AVANT nettoyage - Types de données:")
print("=" * 70)
print(df.dtypes.value_counts().sort_values(ascending=False))

print("\n" + "=" * 70)
print("APRÈS nettoyage - Types de données:")
print("=" * 70)
print(df_clean.dtypes.value_counts().sort_values(ascending=False))

# RÉSUMÉ DES TRANSFORMATIONS - Avant vs Après nettoyage
Dimensions du dataset
 - Avant: (218272, 68) →  Après: (218066, 52)

Lignes supprimées
 - 206 lignes (0.1%)

Colonnes
 - Avant: 68 → Après: 52

Valeurs manquantes (colonnes avec NaN) (normal d'avoir 2, elles sont traitées dans le preprocessing)
 - Avant: 32 → Après: 2

Variable cible (`GRAVITE`)
 - Avant: 5 classes → Après: 3 classes (Materiel, Leger, Grave)

Dates/Heures
 - Avant: chaînes brutes → Après: composantes numérisées (ANNEE, MOIS, JOUR, HEURE, SAISON)

Variables catégorielles
 - Avant: encodées comme float64 → Après: ordonnées ou one-hot encodées
